# What we have done

In [1]:
import numpy as np
import base #templates of functions for MC
import matrix # creates random matrices

We have created a module which gives us a random 2d- or 3d- matrix, which is used for the distance-matrix. The default matrix is symmetric, but we keep the possibility open for an asymmetric matrix, which can be used if e.g. d(1->3) > d(3->1), because salesman have to wait for a train in the other direction.
The 3d- matrix is used, if after a certain time condtions changes (e.g. traffic)

In [22]:
help(matrix.create_testmatrix_3d)

Help on function create_testmatrix_3d in module matrix:

create_testmatrix_3d(high, N, time, symmetric=True, low=1, save='0')
    input: high: highest distance between two points - int or float
            N : number of points of interest -> dimension of matrix _int
            time : number of different time matrices
            symmetric: default True, result will be symmetric matrix
            low: default 1, lowest possible distance _ int or float  
    output: random matrix for testpurpose



For the MC we based our algorithm on "Optimization of the time-dependent traveling salesman problem with Monte Carlo methods". For brevity here is the code for 3d-matrix as input. The only difference to the 2d-matrix is, that we calculate the distance in another way: We used the value of the (round(dist_sum) % periodicity)-matrix for the new entry. This means, that:
If dist_sum =4 and periodicty = 3 we would use the values of the 2nd matrix for this point.

In [ ]:
def traveling_changingDist_MC(points, distances, tries, period, starting_temp = 1, alpha = 0.999 , method =2):
    """ input:  points - points that should be visited by each path (in some order) - array
                distances - 3d-matrix containing the 'distances' and time evolution between each point - 3darray
                tries - itterations of the Monte Carlos - Markov chain - int
                period - times before repetition of matrixpattern after
                starting_temp - scalar in the exponent of the markov chain acception process - int
                alpha - scalar regulating the decrease of the temperature after each loop iteration in the MC
                method - see Opt_method- currently random between Lin-2 & Lin-3
        return: pathWeights - 'distances' (better weights) of each path - array of int
                pathHistory - all paths in the order they have been explored (same as pathWeights) - 2darray
                temps - temperatures after each MC step - array of int
    """
    pathHistory = np.zeros((tries+1, len(points)))
    pathWeights = np.zeros(tries+1)
    temps = np.zeros(tries+1)

    pathWeights[0] = travelWeightTotal_changingDist(points, distances, period)
    #dist_sum += twoPointDist(points[i], points[i+1], distances[:,:,round(dist_sum) % periodicity]
    temps[0] = starting_temp
    cost = pathWeights[0]
    pathHistory[:][0] = points
 
    
    for i in range(tries):
        
        new_sequence = Opt_method(method,points) 
        # Chooses between Lin-2 : exchange of direction and Lin-3 exchange of two parts of the array
        new_cost = travelWeightTotal_changingDist(new_sequence, distances, period)
        if np.random.uniform(low=0, high=1) < np.exp((pathWeights[i]-new_cost)/temps[i]): 
            points = new_sequence
            cost = new_cost
        pathWeights[i+1] = cost
        pathHistory[i+1] = points

        temps[i+1] = alpha*temps[i]  # reduce probability for change 
    return(pathWeights, pathHistory, temps)

As an example we use a small matrix for simplicity and are staying 2d:

In [42]:
A = matrix.create_testmatrix_2d(4,4)
points = np.arange(4)
print(A)

[[0.  1.5 2.  2.5]
 [1.5 0.  2.  2. ]
 [2.  2.  0.  1. ]
 [2.5 2.  1.  0. ]]


In [49]:
base.traveling_MC_constDist(points,A,50, alpha =0.9)[0:2]

(array([2, 3, 1, 0]),
 array([7. , 7. , 7. , 6.5, 6.5, 6.5, 8.5, 6.5, 6.5, 6.5, 6.5, 7. , 7. ,
        7. , 7. , 7. , 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5,
        6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5,
        6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5]))

Our MC-Algorithm is working as intended. We would like to know if our following ideas on further work aligns with the intended ideas.


## Our ideas

* analyse if we get stuck in an local Minima
* show influence of different traffic (3d-matrices) on results